In [1]:
import pandas as pd 
import torch
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, DataCollatorForSeq2Seq, Trainer

In [2]:
train_df=pd.read_csv("samsum-train.csv")
val_df=pd.read_csv("samsum-validation.csv")
test_df=pd.read_csv("samsum-test.csv")

In [3]:
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_df['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_df.shape

(14732, 3)

In [6]:
sample=train_df.iloc[0]

print("Dialogue:\n"+sample["dialogue"])
print("\n"+"="*50+"\n")
print("Summary:",sample["summary"])

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)


Summary: Amanda baked cookies and will bring Jerry some tomorrow.


In [7]:
train_df.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [8]:
train_df=train_df.dropna()
train_df=train_df.reset_index(drop=True)

In [9]:
train_df.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [10]:
train_df.duplicated().sum()

np.int64(0)

In [11]:
train_df['dialogue'].duplicated().sum()

np.int64(467)

In [12]:
train_df[train_df['dialogue'].duplicated()]

,id,dialogue,summary
379,13809997,David: Whats up? \r\nNathan: Nothing much\r\nN...,"Nathan's going to the pool by himself, David d..."
677,13810161,Lawrence: Papa you're at home in the evening? ...,Lawrence will visit Papa tonight after jogging...
829,13611824,Wayne: Help! I need my password for the compan...,Wayne forgot the password for the company shar...
1002,13680728,"Jim: Hey, I've sent you an email about Christm...",Jim is going to come to Poland on the 26th of ...
1573,13715965,Kitty: We don’t have morning class on Tuesday ...,"Kitty, Jim and Kimberly don't have a morning c..."
...,...,...,...
14648,13809899,Agnes: Why don't you send me some pics of your...,Matilde and Marco decorated their house by the...
14655,13810146,Tracey: Never mind diet we really do need catc...,"Tracey and Pauline want to meet to catch up, T..."
14676,13818778,Beth: what are my kids up to? :*\r\nSally: bus...,Beth wants her children Sally and Jake to come...
14707,13716688,Kelly: I need you guys! i'm feeling down! :(\r...,"Kelly is upset, because her boyfriend took her..."


In [13]:
train_df=train_df.drop_duplicates(subset=['dialogue'],keep="first")
train_df=train_df.reset_index(drop=True)

In [14]:
train_df.shape

(14264, 3)

In [15]:
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [16]:
def clean_text(text):
    text=text.replace("\r\n"," ")
    text=text.strip()
    return text

train_df['dialogue']=train_df['dialogue'].apply(clean_text)
val_df['dialogue']=val_df['dialogue'].apply(clean_text)
test_df['dialogue']=test_df['dialogue'].apply(clean_text)

train_df['summary']=train_df['summary'].apply(clean_text)
val_df['summary']=val_df['summary'].apply(clean_text)
test_df['summary']=test_df['summary'].apply(clean_text)

In [17]:
train_dataset=Dataset.from_pandas(train_df)
val_dataset=Dataset.from_pandas(val_df)
test_dataset=Dataset.from_pandas(test_df)

In [18]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [19]:
def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["dialogue"]] # Task prefix for each dialogue
    
    # Tokenize input dialogues (convert text to token IDs)
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target summaries (labels)
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Assign tokenized summaries as labels for training
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [20]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/14264 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [21]:
tokenized_train

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14264
})

In [22]:
tokenized_train[0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.',
 'input_ids': [21603,
  10,
  21542,
  10,
  27,
  13635,
  5081,
  5,
  531,
  25,
  241,
  128,
  58,
  16637,
  10,
  10625,
  55,
  21542,
  10,
  27,
  31,
  195,
  830,
  25,
  5721,
  3,
  10,
  18,
  61,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [23]:
tokenized_train = tokenized_train.remove_columns(["id", "dialogue", "summary"])
tokenized_val = tokenized_val.remove_columns(["id", "dialogue", "summary"])

In [24]:
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14264
})

In [25]:
print(torch.backends.mps.is_available())

True


In [26]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [27]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [28]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    fp16=False
)

In [29]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

In [31]:
trainer.train()

/Users/madhavbhudhiraja/Text Summarizer/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.425394,0.383656
2,0.379867,0.371993
3,0.361605,0.368364


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/madhavbhudhiraja/Text Summarizer/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/madhavbhudhiraja/Text Summarizer/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10698, training_loss=0.39878096110710676, metrics={'train_runtime': 4031.4312, 'train_samples_per_second': 10.615, 'train_steps_per_second': 2.654, 'total_flos': 5791546368589824.0, 'train_loss': 0.39878096110710676, 'epoch': 3.0})

In [32]:
import os

save_path = "./models/t5-samsum-model"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./models/t5-samsum-model/tokenizer_config.json',
 './models/t5-samsum-model/tokenizer.json')

In [33]:
tokenizer=T5Tokenizer.from_pretrained("models/t5-samsum-model")
model=T5ForConditionalGeneration.from_pretrained("models/t5-samsum-model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [36]:
def generate_summary(text):
    text = clean_text(text) # Clean the text

    # Add task prefix for T5
    input_text = "summarize: " + text
    
    # Tokenize text
    inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )
    
    # Generate the summary using the model => token ids 
    outputs = model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    
    # Decode the tokens into text
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return summary

In [38]:
sample = train_df.sample(n=1)["dialogue"].values[0]

print("Dialogue:\n", sample)
print("\nGenerated Summary:\n", generate_summary(sample))

Dialogue:
 Debbie: Do you have time-travel machines in Austria? Lorcan: Haha what the hell. Did you enjoy your weekend? Debbie: I’m the worst driver ever. I feel so sorry about other people in the streets. But besides that, it was ok :D. Lorcan: What did you do? Did you hit someone? Debbie: No, I totally can't use the clutch. So I often stop in the middle of the street and can't move. Cause a car just stops working. Lorcan: Hahaha. That’s dangerous. Automatic gear?  Debbie: I wanted to share a car with my brother, it's a normal one. Mom bought an automatic one a couple of months ago, but it's new, and I bet she wouldn't lend it to me. What did you do??? Lorcan: Haha practice makes perfect! Yesterday and today it was 20+ degrees. I went to Amsterdam with friends. And just chilled in the park with some BBQ. Debbie: Amazing! Perfect weather. No insects yet, not too hot.

Generated Summary:
 Debbie has time-travel machines in Austria. She can't use the clutch and often stops in the middle 

In [39]:
test_dialogue = """
Aman: Hey, are we still on for the group project meeting today?
Riya: Yes, but I might be 10 minutes late.
Kabir: Same here, stuck in traffic.
Aman: No worries. Let's meet at the library at 5:30 then.
Riya: Works for me.
Kabir: Cool, see you both there!
"""

print(generate_summary(test_dialogue))

Aman, Riya and Kabir are meeting at the library at 5:30.


In [40]:
test_dialogue = """
Arjun: Hey guys, did anyone start the presentation for tomorrow?
Sneha: I started the slides but only finished the introduction part.
Rohit: I was supposed to work on the data section, but I got busy with another assignment.
Arjun: That's not good, we don't have much time left.
Sneha: Yeah, deadline is tomorrow morning.
Rohit: I can finish the data analysis tonight and send it over.
Arjun: Great, I'll handle the conclusion and finalize the slides.
Sneha: I'll polish the design and add visuals.
Rohit: Should we meet once before submitting?
Arjun: Let's do a quick call at 10 PM.
Sneha: Works for me.
Rohit: Same here.
Arjun: Okay, let's make sure everything is ready by then.
"""

print(generate_summary(test_dialogue))

Sneha started the presentation for tomorrow. Rohit was supposed to work on the data section, but he got busy with another assignment. Arjun will finish the data analysis tonight and send it over. They will meet at 10 PM.


In [42]:
import evaluate, rouge_score

rouge = evaluate.load("rouge")

In [43]:
def evaluate_model(df, num_samples=100):
    predictions = []
    references = []

    # Take random samples
    sample_df = df.sample(num_samples, random_state=42)

    for _, row in sample_df.iterrows():
        dialogue = row["dialogue"]
        actual_summary = row["summary"]

        generated_summary = generate_summary(dialogue)

        predictions.append(generated_summary)
        references.append(actual_summary)

    # Compute ROUGE scores
    results = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    formatted_results = {
        key: round(float(value), 2) for key, value in results.items()
    }

    return formatted_results

results = evaluate_model(test_df, num_samples=100)

print("ROUGE Evaluation Results:\n")
for metric, score in results.items():
    print(f"{metric.upper():<10}: {score:.2f}")

ROUGE Evaluation Results:

ROUGE1    : 0.49
ROUGE2    : 0.23
ROUGEL    : 0.40
ROUGELSUM : 0.40
